# Kanpur Urban Heat Island (UHI) Thermal Analysis
### Cloud-Native Earth Observation via Google Earth Engine (Landsat 8 & 9 Collection 2 Level-2)

This notebook implements an entirely cloud-based thermal analysis pipeline for **Kanpur, Uttar Pradesh, India**:
- **Zero large downloads** & **Zero GIS software** required on your local machine.
- **30-meter resolution** Landsat 8 & 9 Thermal Infrared Sensor (TIRS) data.
- **USGS standard radiometric calibration**: $T (\text{Kelvin}) = \text{DN} \times 0.00341802 + 149.0$.
- **Concentric microclimate zones**: Urban Core (0–5 km), Suburban ring (5–15 km), Rural ring (15–20 km).
- **NASA POWER / MERRA-2 cross-validation**: Evaluates spatial resolution gains against 50 km coarse reanalysis.

In [ ]:
# Step 1: Initialize Earth Engine
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import os

try:
    ee.Initialize()
    print("✓ Earth Engine initialized successfully!")
except Exception as e:
    print("Earth Engine authentication required. Running ee.Authenticate()...")
    ee.Authenticate()
    ee.Initialize()

## 2. Define Spatial Domain & Concentric Thermal Zones
We center on Kanpur (`26.4499°N, 80.3319°E`) and construct:
1. **Urban Core Zone**: `0 - 5 km` radius (dense built-up core).
2. **Suburban Zone**: `5 - 15 km` annular ring (semi-urban / transitional).
3. **Rural Zone**: `15 - 20 km` annular ring (agricultural / baseline buffer).

In [ ]:
kanpur_lat, kanpur_lon = 26.4499, 80.3319
kanpur_center = ee.Geometry.Point([kanpur_lon, kanpur_lat])
kanpur_buffer = kanpur_center.buffer(20000)  # 20 km total radius

# Define concentric zones as annular rings to prevent spatial contamination
core_buffer = kanpur_center.buffer(5000)      # 0-5 km core
suburban_buffer = kanpur_center.buffer(15000).difference(core_buffer)  # 5-15 km ring
rural_buffer = kanpur_center.buffer(20000).difference(kanpur_center.buffer(15000)) # 15-20 km ring

print("✓ Defined Kanpur spatial domain and annular zones (Core, Suburban, Rural).")

## 3. Query Landsat 8/9 Collection 2 Level-2 Surface Temperature
We query cloud-free scenes ($< 10\%$ cloud cover) from both Landsat 8 and 9.

**USGS Calibration Equation:**
$$\text{LST (°C)} = (\text{DN} \times 0.00341802 + 149.0) - 273.15 = \text{DN} \times 0.00341802 - 124.15$$

In [ ]:
def convert_to_celsius(img):
    lst = img.select('ST_B10') \
        .multiply(0.00341802) \
        .add(149.0) \
        .subtract(273.15) \
        .rename('LST_Celsius')
    return img.addBands(lst).copyProperties(img, ['system:time_start', 'CLOUD_COVER'])

# Landsat 9 + Landsat 8 for complete seasonal coverage
l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2') \
    .filterBounds(kanpur_buffer) \
    .filterDate('2021-07-01', '2022-10-31') \
    .filter(ee.Filter.lt('CLOUD_COVER', 10))

l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(kanpur_buffer) \
    .filterDate('2021-07-01', '2022-10-31') \
    .filter(ee.Filter.lt('CLOUD_COVER', 10))

combined = l9.merge(l8).sort('system:time_start')
thermal_celsius = combined.map(convert_to_celsius)

total_scenes = thermal_celsius.size().getInfo()
print(f"Found {total_scenes} cloud-free Landsat 8/9 scenes over Kanpur.")

## 4. Extract Zonal Statistics & Match with MERRA-2 Benchmark

In [ ]:
# Load local MERRA-2 dataset if available
merra_csv = 'data/raw/kanpur_nasa_power_daily.csv'
merra_dict = {}
if os.path.exists(merra_csv):
    df_m = pd.read_csv(merra_csv)
    if 'Date' in df_m.columns and 'LST_Skin_C' in df_m.columns:
        merra_dict = df_m.set_index('Date')['LST_Skin_C'].to_dict()

results_core = []
results_suburban = []
results_rural = []
merra_matched = []
dates = []

process_count = min(total_scenes, 10)
scene_list = thermal_celsius.toList(process_count)

for i in range(process_count):
    img = ee.Image(scene_list.get(i))
    lst_band = img.select('LST_Celsius')
    
    date_ms = img.get('system:time_start').getInfo()
    date_str = datetime.fromtimestamp(date_ms / 1000, tz=timezone.utc).strftime('%Y-%m-%d')
    
    c_stat = lst_band.reduceRegion(ee.Reducer.mean(), core_buffer, 30, maxPixels=1e9).getInfo()
    s_stat = lst_band.reduceRegion(ee.Reducer.mean(), suburban_buffer, 30, maxPixels=1e9).getInfo()
    r_stat = lst_band.reduceRegion(ee.Reducer.mean(), rural_buffer, 30, maxPixels=1e9).getInfo()
    
    c_val = c_stat.get('LST_Celsius')
    s_val = s_stat.get('LST_Celsius')
    r_val = r_stat.get('LST_Celsius')
    
    if c_val is not None and s_val is not None and r_val is not None:
        results_core.append(c_val)
        results_suburban.append(s_val)
        results_rural.append(r_val)
        dates.append(date_str)
        m_val = merra_dict.get(date_str, np.nan)
        merra_matched.append(m_val)
        
        m_str = f"{m_val:.2f}°C" if not np.isnan(m_val) else "N/A"
        print(f"{date_str}: Core={c_val:5.2f}°C | Suburban={s_val:5.2f}°C | Rural={r_val:5.2f}°C | UHI={c_val-r_val:+5.2f}°C | MERRA-2={m_str}")

## 5. Calculate Urban Heat Island (UHI) Metrics

In [ ]:
mean_core = np.mean(results_core)
mean_sub = np.mean(results_suburban)
mean_rur = np.mean(results_rural)
uhi_intensity = mean_core - mean_rur

print("=== KANPUR UHI SUMMARY ===")
print(f"Urban Core Mean (0-5 km)   : {mean_core:.2f} °C")
print(f"Suburban Mean (5-15 km)    : {mean_sub:.2f} °C")
print(f"Rural Baseline (15-20 km)  : {mean_rur:.2f} °C")
print(f"Net UHI Intensity (ΔT)     : {uhi_intensity:+.2f} °C")

## 6. Plot 1: Seasonal Time Series

In [ ]:
plt.figure(figsize=(11, 5.5))
plt.plot(dates, results_core, marker='o', linewidth=2, color='#d62728', label='Core (0–5 km)')
plt.plot(dates, results_suburban, marker='s', linewidth=1.8, color='#ff7f0e', label='Suburban (5–15 km)')
plt.plot(dates, results_rural, marker='^', linewidth=1.8, color='#2ca02c', label='Rural (15–20 km)')
if any(not np.isnan(v) for v in merra_matched):
    plt.plot(dates, merra_matched, marker='D', linestyle='--', color='#1f77b4', label='MERRA-2 Reanalysis (50 km)')

plt.xlabel('Acquisition Date', fontweight='bold')
plt.ylabel('Land Surface Temperature (°C)', fontweight='bold')
plt.title(f'Kanpur Urban Heat Island: Seasonal Variation (Mean UHI = +{uhi_intensity:.2f}°C)', fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig('kanpur_uhi_timeseries.png', dpi=150)
plt.show()
print("✓ Saved: kanpur_uhi_timeseries.png")

## 7. Plot 2: Concentric Zone Gradient Bar Chart

In [ ]:
zones = ['Core\n(0–5 km)', 'Suburban\n(5–15 km)', 'Rural\n(15–20 km)']
temps = [mean_core, mean_sub, mean_rur]
colors = ['#d9383a', '#f58231', '#3cb44b']

plt.figure(figsize=(7.5, 5))
bars = plt.bar(zones, temps, color=colors, alpha=0.85, edgecolor='black', width=0.5)
plt.ylabel('Mean Surface Temperature (°C)', fontweight='bold')
plt.title(f'Kanpur UHI Intensity: +{uhi_intensity:.2f}°C', fontweight='bold')
plt.ylim([min(temps) - 3, max(temps) + 3])
plt.grid(axis='y', linestyle='--', alpha=0.5)

for bar, temp in zip(bars, temps):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
             f'{temp:.2f}°C', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('kanpur_uhi_zones.png', dpi=150)
plt.show()
print("✓ Saved: kanpur_uhi_zones.png")

## 8. Export Tabular Results to CSV

In [ ]:
df_out = pd.DataFrame({
    'date': dates,
    'core_temp_c': np.round(results_core, 2),
    'suburban_temp_c': np.round(results_suburban, 2),
    'rural_temp_c': np.round(results_rural, 2),
    'uhi_intensity_c': np.round(np.array(results_core) - np.array(results_rural), 2),
    'merra2_skin_temp_c': np.round(merra_matched, 2)
})
df_out.to_csv('kanpur_uhi_results.csv', index=False)
print("✓ Successfully exported kanpur_uhi_results.csv")
display(df_out.head(10))